# HealthBench worst-30% — the entry challenge

Can an open-weights Fusion beat our best open Fusion on the **157 hardest HealthBench
Professional conversations**? This notebook runs the challenge exam end to end: the Engine owns
the Cases, the pinned GPT-5.4 judge (house-paid), the per-rubric-item grading, and the
aggregation; you own the Candidate.

> **This is a challenge metric, NOT an official HealthBench score.** The exam scores the
> team-picked worst-30% subset with an UNCLIPPED mean (official HealthBench clips `max(0, mean)`
> over the full 5,000-row set). On these rows serious baselines score **negative** — that is
> normal, and exactly why the clip is removed: `-0.35 < -0.21 < -0.15` must rank.
>
> Protocol caveats vs the paper: one answer sample (the paper averages 8), no length-adjusted
> score, judge reasoning-effort not yet forwarded, floating `openai/gpt-5.4` slug. The published
> target comes from our own baselines rerun through THIS engine, so comparisons stay fair.

> **Spend warning:** the challenge run is disabled by default. `healthbench/smoke` below is the
> only cell that spends without a switch — one Candidate answer plus one judge call.

## Before running

From a terminal in `packages/screamingface/`:

```bash
just stack-prepare  # first run only: download pinned Benchmark assets
just stack-up       # start AI Gateway :9105 and Engine :9108
just stack-status
```

Use `just stack-logs` to inspect startup failures and `just stack-down` when finished. Stack
management stays outside the notebook so **Run All** never starts or stops local services.

In [1]:
import screamingface as sf

## 1. Connect OpenRouter

In [2]:
sf.connect()

PanelWidget(children=(HTML(value='<style>\n.sf-ui{\n  --sf-bg:#ffffff;--sf-surface:#f6f6f7;--sf-surface-2:#efe…

## 2. Structural smoke — pennies, not comparable

`healthbench/smoke` runs ONE pinned Case (a terse physician research query) through the full
grading chain: Candidate answer → pre-rendered official grader prompt → GPT-5.4 verdict →
unclipped aggregate. Two paid calls; its score is diagnostic only.

In [3]:
# Structural check of the ENTIRE grading chain on one Case (~2 paid calls):
#   - engine serves the case + baked rubric assets (preflight passes)
#   - your Candidate answers through the gateway
#   - the official grader prompt renders and the GPT-5.4 judge returns a
#     parseable verdict
#   - the aggregate scores it: expect `verdict_coverage: 1.0` and a non-null
#     `score` — a null score means a link in the chain failed (see `cases[*]`
#     for the per-Case failure reason)
# The score itself is diagnostic only — NOT comparable to any HealthBench number.
deepseek = sf.Model("openrouter/deepseek/deepseek-v4-pro")
report = sf.evaluate(deepseek, benchmark="healthbench/smoke")

ScreamingFace · Run started
ScreamingFace · Model completed · openrouter/deepseek/deepseek-v4-pro · 9.4s · 27 in / 674 out · stop
ScreamingFace · Model completed · openrouter/openai/gpt-5.4 · 3.8s · 1,272 in / 112 out · stop
ScreamingFace · Run succeeded


In [4]:
report.to_dict()

{'schema': 'screamingface.report.v1',
 'started_at': '2026-08-06T13:49:05.140165Z',
 'completed_at': '2026-08-06T13:49:18.422951Z',
 'benchmark': {'id': 'healthbench/smoke',
  'revision': 'faaf413e0e1a1751',
  'case_count': 1},
 'candidates': [{'run_id': 'V0LyGuw8wsl6VyhCnZ2rwRwGoeGOVqXGKvCo234OiJlCKmxqCXzC0pOoprtXlzfQ',
   'started_at': '2026-08-06T13:49:05.140165Z',
   'completed_at': '2026-08-06T13:49:18.422951Z',
   'name': 'deepseek-v4-pro',
   'kind': 'model',
   'url4': "(candidate:0.0:'(model_1:0.0:/openrouter/deepseek/deepseek-v4-pro?max_tokens=4096&q=($input)!\\'Answer the request accurately and completely. Follow every instruction and formatting constraint in the request.\\')!\\'$model_1\\'', (rows:0.0:(selected_rows:0.0:/benchmarks/healthbench/smoke/faaf413e0e1a1751/cases*(graded:0.0:(rubric_rows:0.0:/benchmarks/healthbench/smoke/faaf413e0e1a1751/rubric-tasks((candidate_result:0.0:/candidate?web_search=false&q=($item.input)!'$candidate')!'$candidate_result')!'$item.id'*(eva

## 3. The challenge Candidate — an open trio

The engine's declared open-weights trio (DeepSeek-V4-Pro, Kimi-K2.6, Qwen3.6-plus — Qwen stands
in for July's GLM-5.2, which has no declared Engine route) with DeepSeek synthesising. Health
answers are graded on accuracy AND safety, so the synthesis policy favours the safer,
better-supported claim and stays concise.

The synthesis prompt below is **yours to change** — it is the baseline's recipe, not part of
the exam protocol. Rewriting it (and swapping panel members or the synthesizer) is the main
experiment surface of this challenge. Members and solo Candidates can carry their own answer
policy too — `sf.Model(..., prompt="...")` — the baseline leaves its members bare only because
the published target was measured that way.

In [6]:
HEALTH_SYNTHESIS_PROMPT = (
    "You are synthesising the single best response to a health-related "
    "conversation by combining independent responses from a panel of models. "
    "A physician-authored rubric will grade your output per criterion — "
    "accuracy, completeness, safety, and appropriate hedging all matter; so "
    "does avoiding harmful or false content.\n\n"
    "Procedure:\n"
    "1. Read every panel response carefully.\n"
    "2. Identify what each contributes that the others miss (facts, safety "
    "caveats, follow-up questions, care-seeking guidance).\n"
    "3. Produce ONE unified response to the final user turn that:\n"
    "   - Keeps every clinically correct, relevant point\n"
    "   - Drops anything inaccurate, unsafe, or speculative\n"
    "   - Resolves disagreements by favouring the safer, better-supported claim\n"
    "   - Stays concise — verbosity is not rewarded\n"
    "4. Do not introduce claims no panel member made.\n\n"
    "Output: the unified response only, no preamble, no JSON wrapper."
)

In [7]:
deepseek = sf.Model("openrouter/deepseek/deepseek-v4-pro")
kimi = sf.Model("openrouter/moonshotai/kimi-k2.6")
qwen = sf.Model("openrouter/qwen/qwen3.6-plus")

open_trio = sf.Fusion(
    [deepseek, kimi, qwen],
    name="open_trio",
    synthesizer="openrouter/deepseek/deepseek-v4-pro",
    prompt=HEALTH_SYNTHESIS_PROMPT,
)

## 4. Arm the challenge run explicitly

157 Cases × (3 members + 1 synthesis) answer calls plus ~350 house-paid judge calls per attempt.
Leaving `RUN_EVALUATION = False` keeps **Run All** free of model spend.

In [ ]:
report = sf.evaluate(
    open_trio, 
    benchmark="healthbench/worst30", 
    limit=2
)

In [9]:
report.to_dict()

{'schema': 'screamingface.report.v1',
 'started_at': '2026-08-06T13:50:55.045536Z',
 'completed_at': '2026-08-06T13:52:28.857974Z',
 'benchmark': {'id': 'healthbench/worst30',
  'revision': '8c8a69413039e67a',
  'case_count': 2},
 'candidates': [{'run_id': 'RQ5EswyP6dnwN1grmWl842DncwhcLQVxcOS0DpaLkQckiwC3p49Ec9pl9avSeNwG',
   'started_at': '2026-08-06T13:50:55.045536Z',
   'completed_at': '2026-08-06T13:52:28.857974Z',
   'name': 'open_trio',
   'kind': 'fusion',
   'url4': "(candidate:0.0:'(model_1:0.0:/openrouter/deepseek/deepseek-v4-pro?max_tokens=4096&q=($input)!\\'Answer the request accurately and completely. Follow every instruction and formatting constraint in the request.\\', model_2:0.0:/openrouter/moonshotai/kimi-k2.6?max_tokens=4096&q=($input)!\\'Answer the request accurately and completely. Follow every instruction and formatting constraint in the request.\\', model_3:0.0:/openrouter/qwen/qwen3.6-plus?max_tokens=4096&q=($input)!\\'Answer the request accurately and completel

## 5. Reading the Report

- `score` — the challenge metric: UNCLIPPED mean over all 157 Case scores. Negative is normal
  here; the number to beat is the published engine-rerun baseline.
- `metrics.verdict_coverage` — must be **1.0** for a valid attempt; any unjudged rubric item
  fails its Case loudly (`score` becomes `null`) rather than silently inflating the mean.
- `metrics.score_sd`, `metrics.judge_invalid_replies` — stability and judge health.
- `cases[*]` — per-Case rubric evidence: every judge verdict with its raw reply, the Candidate's
  exact answer, and the `[points] criterion` lines it was graded against.